In [2]:
import re
from compilerlabs import Tokenizer, TokenizerError, TokenAction


In [3]:
# -----------------------------
# Function: build_json5_tokenizer
# -----------------------------
def build_json5_tokenizer():
    t = Tokenizer()

    # Ignore whitespace and comments
    t.pattern(r'[ \t\n\r]+', TokenAction.IGNORE)
    t.pattern(r'//[^\n\r]*', TokenAction.IGNORE)
    t.pattern(r'/\*[\s\S]*?\*/', TokenAction.IGNORE)

    # Punctuation
    t.pattern(r'\{', 'LBRACE')
    t.pattern(r'\}', 'RBRACE')
    t.pattern(r'\[', 'LBRACKET')
    t.pattern(r'\]', 'RBRACKET')
    t.pattern(r':', 'COLON')
    t.pattern(r',', 'COMMA')

    # Identifiers and keywords
    t.pattern(
        r'[$A-Za-z_][$A-Za-z0-9_]*',
        'IDENTIFIER',
        keywords={'true': 'TRUE', 'false': 'FALSE', 'null': 'NULL'}
    )

    # Numbers
    t.pattern(r'[+-]?0[xX][0-9a-fA-F]+', 'HEXNUMBER')
    t.pattern(r'[+-]?(?:\d+\.\d+|\d+\.|\.\d+|\d+)(?:[eE][+-]?\d+)?', 'NUMBER')

    # Strings
    t.pattern(r'"(?:\\.|[^"\\])*"', 'STRING')
    t.pattern(r"'(?:\\.|[^'\\])*'", 'STRING')

    # Error
    t.pattern(r'.', TokenAction.ERROR)

    return t

In [4]:
# -----------------------------
# Test JSON5 input
# -----------------------------
test_input = r"""
{
  // line comment
  unquotedKey: 'hello',
  "quotedKey": "world",
  age: 25,
  price: -12.5,
  hexValue: 0xFF,
  positive: +10,
  fraction: .75,
  exponent: 1.2e3,
  isActive: true,
  isDeleted: false,
  emptyValue: null,
  list: [1, 2, 3,],
  nested: {
    another_key: "text"
  }
  /* block comment */
}
"""

In [5]:
# -----------------------------
# Run tokenizer
# -----------------------------
t = build_json5_tokenizer()

try:
    for s in t.scan(test_input, eot=False):
        print(f'{s.token:12} | {s.lexeme!r:20} | line={s.lineno}, char={s.charpos}')
except TokenizerError as e:
    print(f'Lexical error: {e.lexeme!r} at line {e.lineno}, char {e.charpos}')

LBRACE       | '{'                  | line=2, char=0
IDENTIFIER   | 'unquotedKey'        | line=4, char=2
COLON        | ':'                  | line=4, char=13
STRING       | "'hello'"            | line=4, char=15
COMMA        | ','                  | line=4, char=22
STRING       | '"quotedKey"'        | line=5, char=2
COLON        | ':'                  | line=5, char=13
STRING       | '"world"'            | line=5, char=15
COMMA        | ','                  | line=5, char=22
IDENTIFIER   | 'age'                | line=6, char=2
COLON        | ':'                  | line=6, char=5
NUMBER       | '25'                 | line=6, char=7
COMMA        | ','                  | line=6, char=9
IDENTIFIER   | 'price'              | line=7, char=2
COLON        | ':'                  | line=7, char=7
NUMBER       | '-12.5'              | line=7, char=9
COMMA        | ','                  | line=7, char=14
IDENTIFIER   | 'hexValue'           | line=8, char=2
COLON        | ':'                  | l

In [6]:
# -----------------------------
# Raw token output (optional)
# -----------------------------
t = build_json5_tokenizer()

try:
    for s in t.scan(test_input, eot=False):
        print(s)
except TokenizerError as e:
    print(f'Lexical error: {e.lexeme!r} at line {e.lineno}, char {e.charpos}')

Symbol(token='LBRACE', lexeme='{', lineno=2, charpos=0)
Symbol(token='IDENTIFIER', lexeme='unquotedKey', lineno=4, charpos=2)
Symbol(token='COLON', lexeme=':', lineno=4, charpos=13)
Symbol(token='STRING', lexeme="'hello'", lineno=4, charpos=15)
Symbol(token='COMMA', lexeme=',', lineno=4, charpos=22)
Symbol(token='STRING', lexeme='"quotedKey"', lineno=5, charpos=2)
Symbol(token='COLON', lexeme=':', lineno=5, charpos=13)
Symbol(token='STRING', lexeme='"world"', lineno=5, charpos=15)
Symbol(token='COMMA', lexeme=',', lineno=5, charpos=22)
Symbol(token='IDENTIFIER', lexeme='age', lineno=6, charpos=2)
Symbol(token='COLON', lexeme=':', lineno=6, charpos=5)
Symbol(token='NUMBER', lexeme='25', lineno=6, charpos=7)
Symbol(token='COMMA', lexeme=',', lineno=6, charpos=9)
Symbol(token='IDENTIFIER', lexeme='price', lineno=7, charpos=2)
Symbol(token='COLON', lexeme=':', lineno=7, charpos=7)
Symbol(token='NUMBER', lexeme='-12.5', lineno=7, charpos=9)
Symbol(token='COMMA', lexeme=',', lineno=7, charpo